M

# Notebook 02: Data Cleaning

**Project:** Who Makes the Canon? Gender Representation in European Painting Collections  
**Author:** Samiulhaq 
**Date:** June 2026

## Purpose
This notebook cleans the raw Europeana data collected in Notebook 01.

## Steps
1. Load raw data
2. Classify creators as named / anonymous / missing
3. Extract year from messy date field
4. Normalise text fields
5. Save clean dataset

## Input
- `../data/raw/europeana_raw.csv`

## Output
- `../data/processed/europeana_clean.csv`

In [6]:
# ============================================================
# IMPORTS AND PATHS
# ============================================================

import pandas as pd
import re
import os

# Input and output paths
INPUT_FILE  = r"C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\raw\europeana_raw.csv"
OUTPUT_FILE = r"C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed\europeana_clean.csv"

os.makedirs(r"C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed", exist_ok=True)

# Load raw data
df = pd.read_csv(INPUT_FILE, encoding="utf-8")

print(f"Raw data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum())

Raw data loaded: 27,454 rows, 12 columns

Columns: ['id', 'title', 'creator', 'date', 'country', 'country_queried', 'provider', 'rights', 'description', 'medium', 'language', 'thumbnail']

Missing values:
id                     0
title                  1
creator                0
date                   0
country                0
country_queried        0
provider               0
rights                 0
description         6062
medium             27454
language               0
thumbnail             68
dtype: int64


In [7]:
# ============================================================
# STEP 1: Classify creators as named / anonymous / missing
# Instead of removing anonymous records we keep them
# as a separate category for analysis
# ============================================================

ANONYMOUS_TERMS = [
    "unknown", "anonymous", "anon", "anoniem", "onbekend",
    "unbekannt", "inconnu", "sconosciuto", "desconocido",
    "attr.", "workshop", "circle of", "follower of",
    "school of", "atelier", "after", "manner of",
    "attributed", "studio", "copy after"
]

def classify_creator(name):
    if pd.isna(name) or str(name).strip() == "":
        return "missing"
    name_lower = str(name).lower().strip()
    for term in ANONYMOUS_TERMS:
        if term in name_lower:
            return "anonymous"
    return "named"

df["creator_type"] = df["creator"].apply(classify_creator)

print("STEP 1 - Creator classification:")
print(df["creator_type"].value_counts())
print(f"\nAs percentages:")
print(df["creator_type"].value_counts(normalize=True).mul(100).round(1))

STEP 1 - Creator classification:
creator_type
named        26360
anonymous     1094
Name: count, dtype: int64

As percentages:
creator_type
named        96.0
anonymous     4.0
Name: proportion, dtype: float64


In [8]:
# ============================================================
# STEP 2: Extract year from messy date field
# Examples of messy dates: "1650", "ca. 1720", "17th century"
# We extract a 4-digit year from whatever is there
# ============================================================

def extract_year(date_string):
    if pd.isna(date_string):
        return None
    date_str = str(date_string)
    matches = re.findall(r'\b(1[2-9]\d{2}|20[0-1]\d)\b', date_str)
    if matches:
        return int(matches[0])
    return None

def year_to_century(year):
    if pd.isna(year):
        return None
    century = ((int(year) - 1) // 100) + 1
    return f"{century}th century"

def year_to_decade(year):
    if pd.isna(year):
        return None
    return int(year) - (int(year) % 10)

df["year_extracted"] = df["date"].apply(extract_year)
df["century"]        = df["year_extracted"].apply(year_to_century)
df["decade"]         = df["year_extracted"].apply(year_to_decade)

print("STEP 2 - Year extraction:")
print(f"Records with year:    {df['year_extracted'].notna().sum():,}")
print(f"Records without year: {df['year_extracted'].isna().sum():,}")
print(f"\nCentury distribution:")
print(df["century"].value_counts().head(10))

STEP 2 - Year extraction:
Records with year:    27,441
Records without year: 13

Century distribution:
century
20th century    12950
19th century     8834
18th century     2626
17th century     1767
16th century      791
15th century      412
14th century       48
13th century        7
21th century        6
Name: count, dtype: int64


In [9]:
# ============================================================
# STEP 3: Clean text fields
# Strip whitespace, standardise capitalisation
# ============================================================

for col in ["country", "provider", "title"]:
    if col in df.columns:
        df[col] = df[col].str.strip().str.title()

print("STEP 3 - Text fields cleaned")
print(f"\nTop 10 countries:")
print(df["country"].value_counts().head(10))
print(f"\nTop 10 providers:")
print(df["provider"].value_counts().head(10))

STEP 3 - Text fields cleaned

Top 10 countries:
country
Sweden     4746
Belgium    3675
Greece     3299
Denmark    2969
Italy      2475
France     2356
Austria    1929
Germany    1859
Spain      1812
Hungary     873
Name: count, dtype: int64

Top 10 providers:
provider
Nationalmuseum Sweden                                                                   2435
Palais Galliera - Musée De La Mode De La Ville De Paris                                 1908
National Gallery Of Denmark                                                             1637
Austrian Gallery Belvedere                                                              1596
The Royal Library: The National Library Of Denmark And Copenhagen University Library    1311
Catholic University Of Leuven                                                           1103
Helsingborg Museum                                                                       960
National Library Of Spain                                                      

In [10]:
# ============================================================
# STEP 4: Add placeholder columns for enrichment
# These will be filled in Notebook 03 via Wikidata
# ============================================================

df["gender"]       = "unknown"
df["wikidata_id"]  = None
df["birth_year"]   = None
df["nationality"]  = None
df["match_status"] = None

print("STEP 4 - Placeholder columns added")
print(f"\nFinal dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nAll columns:")
print(list(df.columns))

STEP 4 - Placeholder columns added

Final dataset shape: 27,454 rows, 21 columns

All columns:
['id', 'title', 'creator', 'date', 'country', 'country_queried', 'provider', 'rights', 'description', 'medium', 'language', 'thumbnail', 'creator_type', 'year_extracted', 'century', 'decade', 'gender', 'wikidata_id', 'birth_year', 'nationality', 'match_status']


In [11]:
# ============================================================
# STEP 5: Save clean dataset
# ============================================================

# Save to processed folder
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("CLEANING SUMMARY")
print("="*50)
print(f"Total records:          {len(df):,}")
print(f"Named artists:          {(df['creator_type']=='named').sum():,}")
print(f"Anonymous:              {(df['creator_type']=='anonymous').sum():,}")
print(f"Records with year:      {df['year_extracted'].notna().sum():,}")
print(f"Records without year:   {df['year_extracted'].isna().sum():,}")
print(f"\nCentury breakdown:")
print(df["century"].value_counts().head(8))
print(f"\nClean data saved to:")
print(OUTPUT_FILE)

# Verify file was saved
if os.path.exists(OUTPUT_FILE):
    size = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
    print(f"\nFile size: {size:.1f} MB")
    print("SUCCESS! Notebook 02 complete!")
else:
    print("ERROR: File not saved!")

CLEANING SUMMARY
Total records:          27,454
Named artists:          26,360
Anonymous:              1,094
Records with year:      27,441
Records without year:   13

Century breakdown:
century
20th century    12950
19th century     8834
18th century     2626
17th century     1767
16th century      791
15th century      412
14th century       48
13th century        7
Name: count, dtype: int64

Clean data saved to:
C:\Users\bfde3\OneDrive\Documents\europeana-gender-painting\data\processed\europeana_clean.csv

File size: 16.8 MB
SUCCESS! Notebook 02 complete!
